# PLQNX Indic QLoRA fine-tuning\n\nThis notebook fine-tunes `dheeyantra/dhee-nxtgen-qwen3-indic` with the PLQNX dataset using 4-bit QLoRA.\n\n**Before running:** choose a GPU runtime. The checked-in dataset is a seed set for pipeline validation, not a production corpus.

In [ ]:
!nvidia-smi\n!git clone https://github.com/jarvis369-max/plqnx-website.git\n%cd plqnx-website\n!pip -q install -r training/requirements.txt

In [ ]:
import torch\nassert torch.cuda.is_available(), 'Enable a GPU runtime first.'\nprint('GPU:', torch.cuda.get_device_name(0))\nprint('BF16 supported:', torch.cuda.is_bf16_supported())

## Build a multilingual training mix\nThis downloads the Apache-2.0 Aya human-annotated instruction dataset, samples PLQNX target languages, and mixes it with the reviewed PLQNX examples.

In [ ]:
!python training/prepare_dataset.py --per-language 75 --custom-repeat 8\n!wc -l training/data/plqnx_mixed_train.jsonl

## Smoke-test training\nRun a short first pass to validate the entire pipeline.

In [ ]:
!python training/train_qlora.py --train-file training/data/plqnx_mixed_train.jsonl --epochs 1 --max-length 1024

## Evaluate the adapter\nThis runs multilingual and coding prompts against the trained adapter.

In [ ]:
!python training/evaluate.py --adapter artifacts/plqnx-indic-4b-lora\n!cat artifacts/eval_comparison.json

## Optional: publish privately to Hugging Face\nRun the login cell, then replace `YOUR_USERNAME` below and upload the adapter.

In [ ]:
from huggingface_hub import notebook_login\nnotebook_login()

In [ ]:
from huggingface_hub import HfApi\napi = HfApi()\nrepo_id = 'YOUR_USERNAME/plqnx-indic-4b-lora'\napi.create_repo(repo_id=repo_id, repo_type='model', private=True, exist_ok=True)\napi.upload_folder(repo_id=repo_id, repo_type='model', folder_path='artifacts/plqnx-indic-4b-lora', commit_message='Upload PLQNX QLoRA adapter')\nprint('Uploaded:', f'https://huggingface.co/{repo_id}')